<a href="https://colab.research.google.com/github/1BF24CS033/BIS/blob/main/Cuckoo_Search_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import math

np.random.seed(42)


bounds = np.array([
    [10.0, 100.0],  # Pt (Watts)
    [30.0, 50.0],   # Gt (dBi)
    [30.0, 50.0]    # Gr (dBi)
])

FREQUENCY = 12e9
DISTANCE = 36000e3
C_SPEED = 3e8
K_DB = -228.6
T_SYS = 290.0
L_ATM = 3.0
C_N0_REQ = 80.0

L_FS = 20 * np.log10(4 * np.pi * DISTANCE * FREQUENCY / C_SPEED) # ~205.15 dB

def objective_function(x):
    Pt_watts, Gt, Gr = x
    Pt_dBW = 10 * np.log10(Pt_watts)
    EIRP = Pt_dBW + Gt
    G_over_T = Gr - 10 * np.log10(T_SYS)
    total_loss = L_FS + L_ATM
    C_N0 = EIRP - total_loss + G_over_T - K_DB
    link_margin = C_N0 - C_N0_REQ
    cost_penalty = 0.05 * Pt_watts + 0.1 * (Gt + Gr)
    fitness = link_margin - cost_penalty

    return fitness, link_margin, C_N0

NUM_NESTS = 50
MAX_ITERATIONS = 100
PA = 0.25  # Discovery probability

dim = len(bounds)
lb = bounds[:, 0]
ub = bounds[:, 1]

nests = np.random.uniform(lb, ub, (NUM_NESTS, dim))
fitness = np.zeros(NUM_NESTS)

for i in range(NUM_NESTS):
    fitness[i], _, _ = objective_function(nests[i])

best_idx = np.argmax(fitness)
best_nest = nests[best_idx].copy()
best_fitness = fitness[best_idx]

def levy_flight(beta=1.5):
    sigma = (math.gamma(1 + beta) * np.sin(np.pi * beta / 2) /
            (math.gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)
    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)
    step = u / (np.abs(v)**(1 / beta))
    return step

print(f"--- STARTING SIMULATION (Nests: {NUM_NESTS}, Iterations: {MAX_ITERATIONS}) ---")
print(f"Initial Best Fitness: {best_fitness:.4f}\n")

for t in range(1, MAX_ITERATIONS + 1):
    for i in range(NUM_NESTS):
        step_length = 0.01 * levy_flight() * (nests[i] - best_nest)
        new_nest = nests[i] + step_length * np.random.normal(0, 1, dim)
        new_nest = np.clip(new_nest, lb, ub)

        f_new, _, _ = objective_function(new_nest)
        if f_new > fitness[i]:
            nests[i] = new_nest
            fitness[i] = f_new

    abandon_mask = np.random.rand(NUM_NESTS) < PA
    for i in range(NUM_NESTS):
        if abandon_mask[i]:
            j, k = np.random.choice(NUM_NESTS, 2, replace=False)
            new_nest = nests[i] + np.random.rand() * (nests[j] - nests[k])
            new_nest = np.clip(new_nest, lb, ub)

            f_new, _, _ = objective_function(new_nest)
            if f_new > fitness[i]:
                nests[i] = new_nest
                fitness[i] = f_new

    current_best_idx = np.argmax(fitness)
    if fitness[current_best_idx] > best_fitness:
        best_fitness = fitness[current_best_idx]
        best_nest = nests[current_best_idx].copy()

    if t in [1, 10, 25, 50, 75, 100]:
        fit_val, margin_val, cn0_val = objective_function(best_nest)
        print(f"Iteration {t:2d} | Fitness: {fit_val:.4f} | Pt: {best_nest[0]:.2f} W | "
              f"Gt: {best_nest[1]:.2f} dBi | Gr: {best_nest[2]:.2f} dBi | "
              f"Link Margin: {margin_val:.2f} dB | C/N0: {cn0_val:.2f} dB-Hz")

print("\n--- OPTIMAL LINK CONFIGURATION FOUND ---")
fit_val, margin_val, cn0_val = objective_function(best_nest)
print(f"Optimal Transmit Power (Pt) : {best_nest[0]:.2f} W ({10*np.log10(best_nest[0]):.2f} dBW)")
print(f"Optimal Tx Antenna Gain (Gt): {best_nest[1]:.2f} dBi")
print(f"Optimal Rx Antenna Gain (Gr): {best_nest[2]:.2f} dBi")
print(f"Carrier-to-Noise Ratio (C/N0): {cn0_val:.2f} dB-Hz")
print(f"Resulting Link Margin        : {margin_val:.2f} dB")

--- STARTING SIMULATION (Nests: 50, Iterations: 100) ---
Initial Best Fitness: 18.8114

Iteration  1 | Fitness: 18.8114 | Pt: 42.73 W | Gt: 49.44 dBi | Gr: 49.25 dBi | Link Margin: 30.82 dB | C/N0: 110.82 dB-Hz
Iteration 10 | Fitness: 20.5279 | Pt: 63.34 W | Gt: 49.84 dBi | Gr: 50.00 dBi | Link Margin: 33.68 dB | C/N0: 113.68 dB-Hz
Iteration 25 | Fitness: 20.8674 | Pt: 89.77 W | Gt: 50.00 dBi | Gr: 50.00 dBi | Link Margin: 35.36 dB | C/N0: 115.36 dB-Hz
Iteration 50 | Fitness: 20.8697 | Pt: 86.57 W | Gt: 50.00 dBi | Gr: 50.00 dBi | Link Margin: 35.20 dB | C/N0: 115.20 dB-Hz
Iteration 75 | Fitness: 20.8697 | Pt: 86.57 W | Gt: 50.00 dBi | Gr: 50.00 dBi | Link Margin: 35.20 dB | C/N0: 115.20 dB-Hz
Iteration 100 | Fitness: 20.8698 | Pt: 86.90 W | Gt: 50.00 dBi | Gr: 50.00 dBi | Link Margin: 35.21 dB | C/N0: 115.21 dB-Hz

--- OPTIMAL LINK CONFIGURATION FOUND ---
Optimal Transmit Power (Pt) : 86.90 W (19.39 dBW)
Optimal Tx Antenna Gain (Gt): 50.00 dBi
Optimal Rx Antenna Gain (Gr): 50.00 dBi
C

In [6]:
import numpy as np

# Analytical optimal values
optimal_Pt = 10.0 / (0.05 * np.log(10)) # Exact peak of 10*log10(Pt) - 0.05*Pt
optimal_Gt = 50.0                     # Upper bound
optimal_Gr = 50.0                     # Upper bound

x_opt = np.array([optimal_Pt, optimal_Gt, optimal_Gr])
max_fitness, margin, cn0 = objective_function(x_opt)

print(f"--- EXACT THEORETICAL OPTIMUM ---")
print(f"Optimal Transmit Power (Pt) : {optimal_Pt:.10f} W")
print(f"Optimal Tx Antenna Gain (Gt): {optimal_Gt:.10f} dBi")
print(f"Optimal Rx Antenna Gain (Gr): {optimal_Gr:.10f} dBi")
print(f"Exact Maximum Fitness        : {max_fitness:.10f}")
print(f"Resulting Link Margin        : {margin:.10f} dB")
print(f"Carrier-to-Noise Ratio (C/N0): {cn0:.10f} dB-Hz")

--- EXACT THEORETICAL OPTIMUM ---
Optimal Transmit Power (Pt) : 86.8588963807 W
Optimal Tx Antenna Gain (Gt): 50.0000000000 dBi
Optimal Rx Antenna Gain (Gr): 50.0000000000 dBi
Exact Maximum Fitness        : 20.8697711493
Resulting Link Margin        : 35.2127159683 dB
Carrier-to-Noise Ratio (C/N0): 115.2127159683 dB-Hz
